# Drone Videolarında İnsan Tespiti — Aşama 2: Test Seti Ön-Etiketleme (Kaggle T4)

Bu notebook **ground-truth test setini** hazırlar: test videolarından seyrek kare örnekler ve
döşemeli (tiled) Grounding DINO öğretmeni ile **yüksek recall** ön-etiket üretir. Çıktı,
CVAT'a yüklenip elle düzeltilecek `annotations.xml` ve değerlendirme için COCO JSON'dur.

## Neden bu notebook Kaggle'da çalışıyor

Yerel GPU (GTX 1650, 4GB) Grounding DINO base ile model geçişi başına ~4.3 saniye harcıyor. 4K
bir kare 10 geçiş gerektirdiği için kare başına ~43 saniye ediyor. Kaggle T4, fp16 tensor
çekirdekleri ve 16GB VRAM ile bu işi yaklaşık bir mertebe hızlandırıyor. Veri seti
(`kmader/drone-videos`) zaten Kaggle'da olduğu için **hiçbir video yüklemek gerekmiyor**.

## Neden döşemeli çıkarım

Grounding DINO'nun image processor'ı girdiyi `shortest_edge=800 / longest_edge=1333`'e
küçültür. 4K (3840x2160) bir kare modele girmeden önce **1333x750'ye, yani 2.88 kat küçülerek**
girer; 60 piksellik bir insan 21 piksele iner ve kaybolur. Kareyi örtüşen parçalara bölmek
etkin çözünürlüğü 4K'da **2.5 kat** artırır.

Etiketleme çevrimdışı olduğu için burada hız değil **recall** önemlidir: insan operatörün işi
"sıfırdan kutu çizmek" değil "fazlalığı silmek" olsun. Bu aynı zamanda ön-etiketleme
yanlılığını azaltır, çünkü operatörün eklemek zorunda kalacağı kutu sayısı düşer.

## Kaggle kurulumu (çalıştırmadan önce)

1. **Settings > Accelerator > GPU T4 x2** (tek T4 yeterli, kod ilk GPU'yu kullanır)
2. **Settings > Internet > On** (Hugging Face'ten model indirilecek)
3. **Add Input > Datasets** > `kmader/drone-videos` ara ve ekle

## Veri bölmesi

Envanter analizine göre 10 videodan 3'ü **video düzeyinde** test setine ayrıldı. Video düzeyinde
bölme şart: aynı videodan hem eğitim hem test karesi alınırsa komşu kareler neredeyse aynı
olduğu için model sahneyi ezberler ve sonuçlar yapay olarak şişer.

| Video | Rejim | Seçim gerekçesi |
|---|---|---|
| `DJI_0596.MP4` | 4K yüksek irtifa | YOLO11x bu videoda **0** tespit yaptı, DINO 62 buldu |
| `Stockflue Flyaround.mp4` | Ultra küçük hedef | Arşivdeki **en yüksek** COCO-small oranı (%60) |
| `Surenen Pass Trail Running.mp4` | Dinamik takip | Patikada koşan sporcu, %49 small |

Kalan 7 video eğitim (pseudo-label) havuzunda kalır ve bu notebook'ta hiç kullanılmaz.

In [ ]:
import subprocess
import sys

import torch
import transformers

print(f"torch        : {torch.__version__}")
print(f"transformers : {transformers.__version__}")
print(f"CUDA         : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU          : {props.name} ({props.total_memory / 1e9:.1f} GB)")
else:
    raise SystemExit("GPU bulunamadi. Settings > Accelerator > GPU T4 x2 secin.")

# Grounding DINO transformers 4.40+ ile geldi. Kaggle imajlari genelde yeterli surumde,
# degilse yukseltiyoruz. Modul kodu her iki API surumuyle de calisir (asagiya bkz).
if tuple(int(p) for p in transformers.__version__.split(".")[:2]) < (4, 40):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers"], check=True)
    print("\ntransformers yukseltildi -> Run > Restart & Run All ile yeniden baslatin.")

## 1. Döşemeli öğretmen modülü

Aşağıdaki hücre `tiled_dino.py` dosyasını yazar. Bu, kod deposundaki modülün **birebir
aynısıdır**; notebook'un tek başına çalışabilmesi için buraya gömülmüştür. Böylece yerel
çalıştırma ile Kaggle çalıştırması aynı döşeme geometrisini ve aynı birleştirme mantığını
kullanır, sonuçlar karşılaştırılabilir kalır.

Birleştirmede iki aşama var çünkü tek başına NMS yetersiz: parça sınırında ikiye bölünen bir
insanın yarım kutusu ile komşu parçadaki tam kutusunun IoU'su düşük kalır ve NMS ikisini de
tutar. Bu yüzden NMS'ten sonra bir **kapsama (containment) bastırması** uygulanır: düşük skorlu
kutu, yüksek skorlu kutunun içinde %80'den fazla kalıyorsa atılır.

In [ ]:
%%writefile tiled_dino.py
"""
Döşemeli (tiled) Grounding DINO çıkarım modülü.

Grounding DINO'nun image processor'ı girdiyi shortest_edge=800 / longest_edge=1333'e
küçültür. 4K (3840x2160) bir drone karesi bu yüzden modele girmeden önce 1333x750'ye,
yani 2.88 kat küçülerek girer: 60 piksellik bir insan 21 piksele iner ve kaybolur.

Bu modül kareyi örtüşen parçalara bölüp her parçayı ayrı ayrı modele vererek etkin
çözünürlük kaybını azaltır, sonra parça sonuçlarını global koordinatlara taşıyıp
NMS + kapsama (containment) bastırması ile birleştirir.

Etiketleme çevrimdışı yapıldığı için burada hız değil recall önemlidir; pahalı
öğretmenin kalitesi daha sonra ucuz bir öğrenci modele damıtılacaktır.
"""

import inspect

import cv2
import numpy as np
import torch
from PIL import Image
from torchvision.ops import nms
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection

DEFAULT_MODEL = "IDEA-Research/grounding-dino-base"

# Grounding DINO image processor'ının hedef boyutları (etkin çözünürlük hesabı için)
PROCESSOR_SHORTEST_EDGE = 800
PROCESSOR_LONGEST_EDGE = 1333


def auto_grid(width, height):
    """
    Çözünürlüğe göre döşeme ızgarasını seçer.

    4K kareler 3x3'e bölünür (parça ~1536x864, yani modele neredeyse tam çözünürlükte
    girer). 720p kareler 2x2'ye bölünür: parçalar processor tarafından bu kez
    büyütülür, bu da küçük insanları belirginleştirir. 406x720 gibi zaten küçük
    kareler tek parça işlenir, bölmek fayda sağlamaz.
    """
    long_side = max(width, height)
    if long_side >= 3000:
        return 3, 3
    if long_side >= 1200:
        return 2, 2
    return 1, 1


def effective_scale(width, height):
    """Bir karenin processor tarafından uygulanan yeniden boyutlandırma oranı."""
    short_side, long_side = min(width, height), max(width, height)
    scale = PROCESSOR_SHORTEST_EDGE / short_side
    if long_side * scale > PROCESSOR_LONGEST_EDGE:
        scale = PROCESSOR_LONGEST_EDGE / long_side
    return scale


def tile_windows(width, height, rows, cols, overlap=0.2):
    """
    Örtüşen ve tümü eşit boyutta olan parça pencereleri üretir.

    Parçaların eşit boyutta olması hem batch'lemeyi güvenli kılar hem de processor'ın
    her parçaya aynı ölçeklemeyi uygulamasını garanti eder. Örtüşme, parça sınırından
    ikiye bölünen bir insanın komşu parçada bütün olarak görünmesini sağlar.
    """
    if rows == 1 and cols == 1:
        return [(0, 0, width, height)]

    tile_w = min(width, int(round(width / cols * (1 + overlap))))
    tile_h = min(height, int(round(height / rows * (1 + overlap))))
    step_x = (width - tile_w) / (cols - 1) if cols > 1 else 0
    step_y = (height - tile_h) / (rows - 1) if rows > 1 else 0

    windows = []
    for r in range(rows):
        for c in range(cols):
            x1 = int(round(c * step_x))
            y1 = int(round(r * step_y))
            windows.append((x1, y1, x1 + tile_w, y1 + tile_h))
    return windows


def geometric_filter(boxes, scores, max_area_px, min_area_px=16.0, max_aspect=4.0):
    """
    Geometrik olarak insan olamayacak kutuları atar.

    Grounding DINO, düz su/gökyüzü/kar gibi dokusuz bölgelerde kesitin tamamını "person"
    olarak kutulama eğiliminde. Bu halüsinasyonlar 0.30+ skor alabildiği, gerçek uzak
    insanlar ise 0.17-0.25 aldığı için eşik yükseltmek işe yaramaz: önce gerçek insanları
    kaybedersin, çöp kalır. Ayırt edici özellik skor değil geometridir.

    max_area_px: mutlak alan üst sınırı. Kasıtlı olarak oran değil piksel: sınır tek bir
        ölçek varsayımından (bir insan bir döşeme parçasının belirli bir oranından büyük
        olamaz) türetilip TÜM geçişlere aynı şekilde uygulanmalı. Tam kare geçişine kare
        oranı uygulanırsa bütçe çok geniş kalır ve tüm sahneyi saran kutular kaçar.
    min_area_px: birkaç piksellik gürültü kutuları atılır
    max_aspect:  insan bu kadar kat "geniş" olamaz (w/h üst sınırı)
    """
    if len(boxes) == 0:
        return boxes, scores

    widths = boxes[:, 2] - boxes[:, 0]
    heights = boxes[:, 3] - boxes[:, 1]
    areas = widths * heights

    keep = (
        (areas <= max_area_px)
        & (areas >= min_area_px)
        & (widths > 0)
        & (heights > 0)
        & (widths <= heights * max_aspect)
    )
    return boxes[keep], scores[keep]


def suppress_contained(boxes, scores, containment_thresh=0.80):
    """
    Düşük skorlu bir kutu, yüksek skorlu bir kutunun içinde büyük oranda kalıyorsa atar.

    NMS bunu yakalayamaz: parça sınırında kesilen bir insanın yarım kutusu ile tam
    kutusunun IoU'su düşük kalır, ama yarım kutu tam kutunun içindedir.
    """
    if len(boxes) == 0:
        return boxes, scores

    order = np.argsort(-scores)
    boxes, scores = boxes[order], scores[order]
    areas = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
    keep = np.ones(len(boxes), dtype=bool)

    for i in range(len(boxes)):
        if not keep[i]:
            continue
        for j in range(i + 1, len(boxes)):
            if not keep[j] or areas[j] <= 0:
                continue
            ix1 = max(boxes[i, 0], boxes[j, 0])
            iy1 = max(boxes[i, 1], boxes[j, 1])
            ix2 = min(boxes[i, 2], boxes[j, 2])
            iy2 = min(boxes[i, 3], boxes[j, 3])
            inter = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
            if inter / areas[j] >= containment_thresh:
                keep[j] = False

    return boxes[keep], scores[keep]


class TiledGroundingDino:
    """Donmuş Grounding DINO öğretmeni: tam kare ve döşemeli çıkarım sağlar."""

    def __init__(self, model_name=DEFAULT_MODEL, device=None, use_fp16=False,
                 text_prompt="person."):
        """
        use_fp16 varsayılan olarak kapalıdır. GTX 1650 (Turing TU117) üzerinde fp16,
        406x720 gibi sıra dışı en-boy oranlarında hata vermeden SIFIR tespit döndürdü
        (fp32 aynı karelerde 0.77 skorla insan buluyordu). Sessiz başarısızlık veri
        üretim hattında en tehlikeli hata türü olduğu için varsayılan fp32'dir; fp16'yı
        açmadan önce iki dtype'ı aynı kareler üzerinde karşılaştırıp doğrulayın.
        """
        self.model_name = model_name
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.dtype = torch.float16 if (use_fp16 and self.device == "cuda") else torch.float32
        self.text_prompt = text_prompt

        print(f"{model_name} yükleniyor... (cihaz: {self.device}, dtype: {self.dtype})")
        self.processor = AutoProcessor.from_pretrained(model_name)
        # transformers v5 `dtype`, v4 `torch_dtype` bekliyor; Kaggle/Colab sürümleri farklı
        try:
            self.model = AutoModelForZeroShotObjectDetection.from_pretrained(
                model_name, dtype=self.dtype
            )
        except TypeError:
            self.model = AutoModelForZeroShotObjectDetection.from_pretrained(
                model_name, torch_dtype=self.dtype
            )
        self.model = self.model.to(self.device).eval()

        # v5 `threshold`, v4 `box_threshold` kullanıyor
        post_process_params = inspect.signature(
            self.processor.post_process_grounded_object_detection
        ).parameters
        self._box_thresh_kwarg = "threshold" if "threshold" in post_process_params else "box_threshold"

    def _forward(self, crops_bgr, threshold, text_threshold):
        """Bir grup BGR kesiti modelden geçirip her biri için (boxes, scores) döndürür."""
        pil_images = [Image.fromarray(cv2.cvtColor(c, cv2.COLOR_BGR2RGB)) for c in crops_bgr]
        inputs = self.processor(
            images=pil_images,
            text=[self.text_prompt] * len(pil_images),
            return_tensors="pt",
        ).to(self.device)
        if self.dtype == torch.float16:
            inputs["pixel_values"] = inputs["pixel_values"].half()

        with torch.inference_mode():
            outputs = self.model(**inputs)

        results = self.processor.post_process_grounded_object_detection(
            outputs,
            input_ids=inputs["input_ids"],
            text_threshold=text_threshold,
            target_sizes=[(c.shape[0], c.shape[1]) for c in crops_bgr],
            **{self._box_thresh_kwarg: threshold},
        )
        return [
            (
                r["boxes"].float().cpu().numpy().reshape(-1, 4),
                r["scores"].float().cpu().numpy().reshape(-1),
            )
            for r in results
        ]

    def detect(self, frame_bgr, threshold=0.25, text_threshold=0.25):
        """Baseline davranışı: tek geçişte tam kare çıkarımı."""
        return self._forward([frame_bgr], threshold, text_threshold)[0]

    def detect_tiled(self, frame_bgr, grid=None, overlap=0.2, threshold=0.15,
                     text_threshold=0.15, iou_thresh=0.55, include_full_frame=True,
                     max_area_ratio=0.25, batch_size=1):
        """
        Döşemeli çıkarım: parçalar + (opsiyonel) tam kare, ardından NMS ile birleştirme.

        Tam kare de dahil edilir çünkü parçalara sığmayan büyük/yakın insanları yakalar;
        parçalar ise küçük/uzak insanları yakalar. İkisi birbirini tamamlar.

        Geometrik filtre BİRLEŞTİRMEDEN ÖNCE, her geçişe aynı mutlak alan sınırıyla
        uygulanır. Sınır parça boyutundan türetilir: bir insan bir parçanın
        max_area_ratio oranından büyük olamaz. Aynı sınır tam kare geçişine de uygulanır,
        aksi halde tüm sahneyi saran halüsinasyon kutuları süzgeçten kaçar.
        """
        h, w = frame_bgr.shape[:2]
        rows, cols = grid or auto_grid(w, h)
        windows = tile_windows(w, h, rows, cols, overlap)

        tile_w = windows[0][2] - windows[0][0]
        tile_h = windows[0][3] - windows[0][1]
        max_area_px = tile_w * tile_h * max_area_ratio

        crops, offsets = [], []
        for (x1, y1, x2, y2) in windows:
            crops.append(frame_bgr[y1:y2, x1:x2])
            offsets.append((x1, y1))
        if include_full_frame and (rows, cols) != (1, 1):
            crops.append(frame_bgr)
            offsets.append((0, 0))

        all_boxes, all_scores = [], []
        for start in range(0, len(crops), batch_size):
            chunk = crops[start:start + batch_size]
            chunk_offsets = offsets[start:start + batch_size]
            for (boxes, scores), (ox, oy) in zip(
                self._forward(chunk, threshold, text_threshold), chunk_offsets
            ):
                boxes, scores = geometric_filter(boxes, scores, max_area_px)
                if len(boxes) == 0:
                    continue
                boxes = boxes.copy()
                boxes[:, [0, 2]] += ox
                boxes[:, [1, 3]] += oy
                all_boxes.append(boxes)
                all_scores.append(scores)

        if not all_boxes:
            return np.zeros((0, 4), dtype=np.float32), np.zeros((0,), dtype=np.float32)

        boxes = np.clip(np.concatenate(all_boxes), [0, 0, 0, 0], [w, h, w, h]).astype(np.float32)
        scores = np.concatenate(all_scores).astype(np.float32)

        keep = nms(torch.from_numpy(boxes), torch.from_numpy(scores), iou_thresh).numpy()
        boxes, scores = boxes[keep], scores[keep]
        return suppress_contained(boxes, scores)


def tiling_report(width, height, overlap=0.2):
    """Bir çözünürlük için döşemenin etkin çözünürlük kazancını raporlar."""
    rows, cols = auto_grid(width, height)
    windows = tile_windows(width, height, rows, cols, overlap)
    tw, th = windows[0][2] - windows[0][0], windows[0][3] - windows[0][1]
    return {
        "resolution": f"{width}x{height}",
        "grid": f"{rows}x{cols}",
        "tile_size": f"{tw}x{th}",
        "forward_passes": len(windows) + (1 if (rows, cols) != (1, 1) else 0),
        "fullframe_scale": round(effective_scale(width, height), 3),
        "tile_scale": round(effective_scale(tw, th), 3),
        "resolution_gain": round(effective_scale(tw, th) / effective_scale(width, height), 2),
    }


def draw_detections(frame, boxes, scores, color=(0, 255, 0), label_prefix="person"):
    """Kutuları kare üzerine çizer; çözünürlükle ölçeklenen kalınlık/font kullanır."""
    annotated = frame.copy()
    h, w = annotated.shape[:2]
    thickness = max(1, round((h + w) / 1600))
    font_scale = max(0.4, thickness * 0.4)

    for (x1, y1, x2, y2), score in zip(boxes, scores):
        p1, p2 = (int(x1), int(y1)), (int(x2), int(y2))
        cv2.rectangle(annotated, p1, p2, color, thickness)
        label = f"{label_prefix} {score:.2f}"
        (tw, th), baseline = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, font_scale, thickness)
        top = p1[1] - th - baseline
        if top < 0:
            top = p1[1] + th + baseline
        cv2.rectangle(annotated, (p1[0], top - th - baseline), (p1[0] + tw, top), color, -1)
        cv2.putText(annotated, label, (p1[0], top - baseline), cv2.FONT_HERSHEY_SIMPLEX,
                    font_scale, (0, 0, 0), thickness, cv2.LINE_AA)
    return annotated


if __name__ == "__main__":
    # Arşivdeki çözünürlükler için döşeme planını ve çözünürlük kazancını yazdır
    print(f"{'Çözünürlük':>12} | {'Izgara':>6} | {'Parça':>10} | {'Geçiş':>5} | "
          f"{'Tam kare':>8} | {'Parça':>6} | {'Kazanç':>6}")
    print("-" * 72)
    for width, height in [(3840, 2160), (1280, 720), (720, 1280), (406, 720)]:
        r = tiling_report(width, height)
        print(f"{r['resolution']:>12} | {r['grid']:>6} | {r['tile_size']:>10} | "
              f"{r['forward_passes']:>5} | {r['fullframe_scale']:>8} | {r['tile_scale']:>6} | "
              f"{r['resolution_gain']:>5}x")

## 2. Videoları bul ve test bölmesini doğrula

Kaggle dataset'inin klasör yapısı sürüme göre değişebildiği için videoları sabit bir yola
güvenmek yerine `/kaggle/input` altında **isimle arıyoruz**. Böylece dataset nasıl paketlenmiş
olursa olsun notebook çalışır.

In [ ]:
import os

import cv2

from tiled_dino import tiling_report

VIDEO_EXTS = (".mp4", ".avi", ".mov", ".mkv")
KAGGLE_INPUT = "/kaggle/input"

# Envanter analizine gore test setine ayrilan videolar ve secim gerekceleri
TEST_VIDEOS = {
    "DJI_0596.MP4": "4K yuksek irtifa - YOLO11x bu videoda 0 tespit yapti, DINO 62 buldu",
    "Stockflue Flyaround.mp4": "En yuksek kucuk-nesne orani (COCO small %60)",
    "Surenen Pass Trail Running.mp4": "Dinamik takip - patikada kosan sporcu (%49 small)",
}


def find_videos(root):
    """root altindaki tum videolari {dosya adi: tam yol} olarak dondurur."""
    found = {}
    for dirpath, _, filenames in os.walk(root):
        for name in filenames:
            if name.lower().endswith(VIDEO_EXTS):
                found.setdefault(name, os.path.join(dirpath, name))
    return found


available = find_videos(KAGGLE_INPUT)
print(f"{KAGGLE_INPUT} altinda {len(available)} video bulundu:")
for name in sorted(available):
    print(f"  {name}")

missing = [n for n in TEST_VIDEOS if n not in available]
if missing:
    raise SystemExit(
        f"\nTest videolari bulunamadi: {missing}\n"
        "Add Input > Datasets > 'kmader/drone-videos' ekledigininzden emin olun."
    )

print("\n" + "=" * 100)
print("TEST BOLMESI (bu kareler egitimde kullanilmayacak)")
print("=" * 100)
test_paths = {}
for name, reason in TEST_VIDEOS.items():
    path = available[name]
    cap = cv2.VideoCapture(path)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    if width == 0:
        raise SystemExit(f"{name} okunamadi ({path})")

    report = tiling_report(width, height)
    test_paths[name] = path
    print(f"\n{name}  ({width}x{height}, {total} kare)")
    print(f"  Gerekce  : {reason}")
    print(f"  Dosem    : {report['grid']} -> parca {report['tile_size']}, "
          f"{report['forward_passes']} model gecisi/kare")
    print(f"  Cozunurluk: tam kare {report['fullframe_scale']}x -> parca "
          f"{report['tile_scale']}x  (kazanc {report['resolution_gain']}x)")

train_videos = sorted(set(available) - set(TEST_VIDEOS))
print(f"\nEgitim havuzunda kalan {len(train_videos)} video: {train_videos}")

## 3. Kareleri örnekle

Kareler **saniyede bir** (her 30. kare) alınır. 30 FPS'de ardışık iki kare arasında 33
milisaniye vardır: koşan bir insan 2-3 piksel yer değiştirir, ışık ve arka plan aynıdır. Yani
komşu kareler bağımsız veri noktası değil, aynı bilginin kopyasıdır. Her kareyi etiketlemek
manuel maliyeti 30 katına çıkarırken test setinin bilgi içeriğini artırmaz; üstelik modelin tek
bir karede yaptığı hata 30 kez sayılacağı için metrikleri de çarpıtır.

Okuma sıralı yapılır: `CAP_PROP_POS_FRAMES` ile atlama 4K akışlarda yavaş olabiliyor.

In [ ]:
SAMPLE_INTERVAL = 30       # her 30. kare = saniyede 1 kare
JPEG_QUALITY = 95          # kucuk insanlarda sikistirma bozulmasi olmasin

OUT_ROOT = "/kaggle/working/dataset/test"
IMAGES_DIR = os.path.join(OUT_ROOT, "images")
ANN_DIR = os.path.join(OUT_ROOT, "annotations")
for d in (IMAGES_DIR, ANN_DIR):
    os.makedirs(d, exist_ok=True)


def slugify(video_name):
    stem = os.path.splitext(video_name)[0]
    return "".join(ch if ch.isalnum() else "_" for ch in stem).strip("_")


def sample_frames(video_path, video_name, interval, out_dir):
    """Videoyu sirali okuyup her `interval` karede birini diske yazar."""
    cap = cv2.VideoCapture(video_path)
    slug = slugify(video_name)
    records, frame_idx = [], 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if frame_idx % interval == 0:
            file_name = f"{slug}_f{frame_idx:06d}.jpg"
            cv2.imwrite(os.path.join(out_dir, file_name), frame,
                        [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])
            records.append({
                "file_name": file_name, "video": video_name, "frame_index": frame_idx,
                "width": frame.shape[1], "height": frame.shape[0],
            })
        frame_idx += 1
    cap.release()
    return records


frame_records = []
for name, path in test_paths.items():
    records = sample_frames(path, name, SAMPLE_INTERVAL, IMAGES_DIR)
    frame_records.extend(records)
    print(f"{name:<34} -> {len(records):3d} kare")

# CVAT bir gorsel klasorunu alfabetik siralar; ayni sirayi kullaniyoruz
frame_records.sort(key=lambda r: r["file_name"])
print(f"\nToplam {len(frame_records)} kare -> {IMAGES_DIR}")

## 4. fp16 güvenlik kontrolü

fp16 bu iş yükünü kabaca iki katına hızlandırır, ama **sessizce bozulabiliyor**: yerel GTX 1650
üzerinde 406x720 karelerde fp16 hiç hata vermeden sıfır tespit döndürdü, aynı karelerde fp32
0.77 skorla insan buluyordu. Veri üretim hattında en tehlikeli hata türü budur, çünkü boş
etiketler fark edilmeden eğitim setine akar.

Bu yüzden fp16'yı varsaymak yerine **ölçüyoruz**: birkaç kare üzerinde fp16 ve fp32 tespit
sayılarını karşılaştırıp, ancak uyuşuyorsa fp16 ile devam ediyoruz.

In [ ]:
import gc
import time

import numpy as np
import torch

from tiled_dino import TiledGroundingDino

BOX_THRESHOLD = 0.15   # DUSUK esik: on-etiketlemede recall onceligi
TEXT_THRESHOLD = 0.15
TEXT_PROMPT = "person."


def probe_frames(records, per_video=2):
    """Her test videosundan birkac temsili kare secer (fp16 kontrolu icin)."""
    chosen = []
    for video in TEST_VIDEOS:
        subset = [r for r in records if r["video"] == video]
        if not subset:
            continue
        step = max(1, len(subset) // (per_video + 1))
        chosen.extend(subset[step::step][:per_video])
    return chosen


probes = probe_frames(frame_records)
probe_images = [cv2.imread(os.path.join(IMAGES_DIR, r["file_name"])) for r in probes]
print(f"{len(probes)} kare uzerinde fp16/fp32 karsilastirmasi\n")

counts = {}
for use_fp16 in (False, True):
    teacher = TiledGroundingDino(use_fp16=use_fp16, text_prompt=TEXT_PROMPT)
    tag = "fp16" if use_fp16 else "fp32"
    counts[tag] = []
    start = time.time()
    for image in probe_images:
        boxes, _ = teacher.detect_tiled(image, threshold=BOX_THRESHOLD,
                                        text_threshold=TEXT_THRESHOLD)
        counts[tag].append(len(boxes))
    counts[tag + "_time"] = time.time() - start
    del teacher
    gc.collect()
    torch.cuda.empty_cache()

print(f"\n{'Kare':<40} | {'fp32':>5} | {'fp16':>5}")
print("-" * 56)
for record, n32, n16 in zip(probes, counts["fp32"], counts["fp16"]):
    print(f"{record['file_name']:<40} | {n32:>5} | {n16:>5}")

total32, total16 = sum(counts["fp32"]), sum(counts["fp16"])
speedup = counts["fp32_time"] / max(counts["fp16_time"], 1e-6)
# fp16 toplam tespitin %90'inin altina duserse guvenilmez kabul edilir
agrees = total16 >= 0.90 * total32 and total32 > 0
USE_FP16 = bool(agrees)

print(f"\nfp32 toplam {total32} kutu ({counts['fp32_time']:.1f}s)")
print(f"fp16 toplam {total16} kutu ({counts['fp16_time']:.1f}s, {speedup:.2f}x hizli)")
print(f"\nKARAR: {'fp16 kullanilacak' if USE_FP16 else 'fp32 kullanilacak (fp16 guvenilmez)'}")

## 5. Ön-etiketleri üret

Karşılaştırma yapabilmek için her kare **iki kez** işlenir: bir kez tam kare (baseline
davranışı), bir kez döşemeli. Böylece "döşeme kaç insan kazandırdı" sorusunu ön-etiket
aşamasında bile sayısal olarak cevaplayabiliyoruz; bu sayı raporun "başlangıç yaklaşımının
problemleri" bölümüne doğrudan girecek.

CVAT'a yüklenecek olan **döşemeli** sonuçlardır (yüksek recall).

In [ ]:
TILE_BATCH_SIZE = 4   # T4'un 16GB'i rahat kaldirir; OOM alirsan 1-2'ye dusur

teacher = TiledGroundingDino(use_fp16=USE_FP16, text_prompt=TEXT_PROMPT)

start = time.time()
for i, record in enumerate(frame_records, 1):
    frame = cv2.imread(os.path.join(IMAGES_DIR, record["file_name"]))

    full_boxes, _ = teacher.detect(frame, BOX_THRESHOLD, TEXT_THRESHOLD)
    boxes, scores = teacher.detect_tiled(
        frame, threshold=BOX_THRESHOLD, text_threshold=TEXT_THRESHOLD,
        batch_size=TILE_BATCH_SIZE,
    )

    record["boxes"] = boxes.tolist()
    record["scores"] = scores.tolist()
    record["fullframe_count"] = int(len(full_boxes))

    if i % 10 == 0 or i == len(frame_records):
        elapsed = time.time() - start
        remaining = (len(frame_records) - i) * elapsed / i
        print(f"  {i:3d}/{len(frame_records)} kare | {elapsed / i:.2f} s/kare | "
              f"kalan ~{remaining / 60:.1f} dk")

print(f"\nTamamlandi: {time.time() - start:.0f} saniye")

## 6. Döşemenin kazancı: tam kare ile karşılaştırma

Bu tablo, alternatif yaklaşımın gerekçesini doğrudan sayıya döküyor. Beklenti: kazanç 4K
videoda (`DJI_0596`) en yüksek, 406x720 videoda (`Surenen`) sıfır, çünkü o çözünürlükte
processor kareyi zaten büyüttüğü için bölmenin faydası yok.

In [ ]:
per_video = {}
for record in frame_records:
    stats = per_video.setdefault(record["video"], {
        "frames": 0, "empty": 0, "tiled": 0, "full": 0, "heights": [], "areas": [],
    })
    stats["frames"] += 1
    stats["tiled"] += len(record["boxes"])
    stats["full"] += record["fullframe_count"]
    if not record["boxes"]:
        stats["empty"] += 1
    for x1, y1, x2, y2 in record["boxes"]:
        stats["heights"].append(y2 - y1)
        stats["areas"].append((x2 - x1) * (y2 - y1))

header = (f"{'Video':<32} | {'Kare':>4} | {'Bos':>4} | {'Tam kare':>8} | {'Dosemeli':>8} | "
          f"{'Kazanc':>7} | {'Medyan h':>8} | {'Small %':>7}")
print(header)
print("-" * len(header))
for video, stats in per_video.items():
    gain = stats["tiled"] / stats["full"] if stats["full"] else float("inf")
    median_h = np.median(stats["heights"]) if stats["heights"] else 0
    small_pct = 100 * np.mean([a < 32 * 32 for a in stats["areas"]]) if stats["areas"] else 0
    print(f"{video:<32} | {stats['frames']:>4} | {stats['empty']:>4} | {stats['full']:>8} | "
          f"{stats['tiled']:>8} | {gain:>6.2f}x | {median_h:>7.1f}px | {small_pct:>6.1f}%")

total_full = sum(s["full"] for s in per_video.values())
total_tiled = sum(s["tiled"] for s in per_video.values())
print("-" * len(header))
print(f"{'TOPLAM':<32} | {len(frame_records):>4} | "
      f"{sum(s['empty'] for s in per_video.values()):>4} | {total_full:>8} | {total_tiled:>8} | "
      f"{total_tiled / max(total_full, 1):>6.2f}x |")
print(f"\nDoseme sayesinde {total_tiled - total_full} ek aday kutu bulundu "
      f"(+%{100 * (total_tiled - total_full) / max(total_full, 1):.0f}).")
print("Bu kutular henuz DOGRULANMAMISTIR; bir kismi yanlis pozitiftir ve CVAT'ta silinecektir.")

## 7. CVAT ve COCO formatlarına aktar

İki format üretiliyor:

- **CVAT for images 1.1 XML** — elle düzeltme için. CVAT eşleştirmeyi `<image name>` alanı
  üzerinden yaptığı için görsel dosya adlarının birebir aynı olması gerekir.
- **COCO JSON** — değerlendirme altyapısı için. `pycocotools` bu formatı doğrudan okur ve
  `AP_small` / `AP_medium` / `AP_large` kırılımını standart tanımlarla verir; baseline'ın küçük
  nesnelerdeki zayıflığını tam olarak bu metrik ortaya çıkaracak.

Kutular `source="auto"` ile işaretleniyor, böylece CVAT'ta hangi kutunun makine üretimi olduğu
görünür kalıyor.

In [ ]:
import json
import zipfile
from datetime import datetime, timezone
from xml.etree import ElementTree as ET

CVAT_LABEL = "person"
CVAT_LABEL_COLOR = "#33ddff"


def build_cvat_xml(images, task_name):
    """CVAT for images 1.1 XML uretir."""
    now = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S.%f+00:00")
    root = ET.Element("annotations")
    ET.SubElement(root, "version").text = "1.1"

    meta = ET.SubElement(root, "meta")
    task = ET.SubElement(meta, "task")
    for tag, value in [("id", "1"), ("name", task_name), ("size", str(len(images))),
                       ("mode", "annotation"), ("overlap", "0")]:
        ET.SubElement(task, tag).text = value
    ET.SubElement(task, "bugtracker")
    ET.SubElement(task, "created").text = now
    ET.SubElement(task, "updated").text = now
    ET.SubElement(task, "start_frame").text = "0"
    ET.SubElement(task, "stop_frame").text = str(max(0, len(images) - 1))
    ET.SubElement(task, "frame_filter")

    labels = ET.SubElement(task, "labels")
    label = ET.SubElement(labels, "label")
    ET.SubElement(label, "name").text = CVAT_LABEL
    ET.SubElement(label, "color").text = CVAT_LABEL_COLOR
    ET.SubElement(label, "type").text = "rectangle"
    ET.SubElement(label, "attributes")

    segments = ET.SubElement(task, "segments")
    segment = ET.SubElement(segments, "segment")
    ET.SubElement(segment, "id").text = "1"
    ET.SubElement(segment, "start").text = "0"
    ET.SubElement(segment, "stop").text = str(max(0, len(images) - 1))
    ET.SubElement(segment, "url")

    owner = ET.SubElement(task, "owner")
    ET.SubElement(owner, "username")
    ET.SubElement(owner, "email")
    ET.SubElement(meta, "dumped").text = now

    for image_id, item in enumerate(images):
        img_el = ET.SubElement(root, "image", {
            "id": str(image_id), "name": item["file_name"],
            "width": str(item["width"]), "height": str(item["height"]),
        })
        for (x1, y1, x2, y2), score in zip(item["boxes"], item["scores"]):
            ET.SubElement(img_el, "box", {
                "label": CVAT_LABEL, "source": "auto", "occluded": "0",
                "xtl": f"{x1:.2f}", "ytl": f"{y1:.2f}",
                "xbr": f"{x2:.2f}", "ybr": f"{y2:.2f}",
                "z_order": "0", "score": f"{score:.4f}",
            })

    ET.indent(root, space="  ")
    return ET.ElementTree(root)


def build_coco(images, description):
    coco = {
        "info": {"description": description,
                 "date_created": datetime.now(timezone.utc).isoformat()},
        "licenses": [], "images": [], "annotations": [],
        "categories": [{"id": 1, "name": CVAT_LABEL, "supercategory": "person"}],
    }
    ann_id = 1
    for image_id, item in enumerate(images):
        coco["images"].append({
            "id": image_id, "file_name": item["file_name"],
            "width": item["width"], "height": item["height"],
            "video": item["video"], "frame_index": item["frame_index"],
        })
        for (x1, y1, x2, y2), score in zip(item["boxes"], item["scores"]):
            w, h = float(x2 - x1), float(y2 - y1)
            coco["annotations"].append({
                "id": ann_id, "image_id": image_id, "category_id": 1,
                "bbox": [round(float(x1), 2), round(float(y1), 2), round(w, 2), round(h, 2)],
                "area": round(w * h, 2), "iscrowd": 0, "score": round(float(score), 4),
            })
            ann_id += 1
    return coco


xml_path = os.path.join(ANN_DIR, "test_preannot_cvat.xml")
build_cvat_xml(frame_records, "drone_person_test_gt").write(
    xml_path, encoding="utf-8", xml_declaration=True)

cvat_zip = os.path.join(ANN_DIR, "test_preannot_cvat.zip")
with zipfile.ZipFile(cvat_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(xml_path, "annotations.xml")

coco_path = os.path.join(ANN_DIR, "test_preannot_coco.json")
with open(coco_path, "w", encoding="utf-8") as f:
    json.dump(build_coco(frame_records, "Drone person detection - dosemeli DINO on-etiketleri"),
              f, indent=2, ensure_ascii=False)

# Kare indeksi: hangi kare hangi videonun kacinci karesi (izlenebilirlik icin)
index_path = os.path.join(OUT_ROOT, "frame_index.json")
with open(index_path, "w", encoding="utf-8") as f:
    json.dump({
        "split": "test",
        "note": "Bu kareler egitimde kullanilmaz.",
        "sample_interval": SAMPLE_INTERVAL,
        "preannotation": {
            "model": teacher.model_name, "text_prompt": TEXT_PROMPT,
            "box_threshold": BOX_THRESHOLD, "text_threshold": TEXT_THRESHOLD,
            "tiled": True, "fp16": USE_FP16,
        },
        "per_video": {v: {"frames": s["frames"], "tiled_boxes": s["tiled"],
                          "fullframe_boxes": s["full"]} for v, s in per_video.items()},
        "frames": [{k: v for k, v in r.items() if k not in ("boxes", "scores")}
                   for r in frame_records],
    }, f, indent=2, ensure_ascii=False)

# Gorselleri tek zip'te indirilebilir yap (CVAT'a yuklemek icin)
images_zip = "/kaggle/working/test_images.zip"
with zipfile.ZipFile(images_zip, "w", zipfile.ZIP_STORED) as zf:
    for record in frame_records:
        zf.write(os.path.join(IMAGES_DIR, record["file_name"]), record["file_name"])

for path in (images_zip, cvat_zip, coco_path, index_path):
    print(f"{os.path.getsize(path) / 1e6:8.1f} MB  {path}")

## 8. Görsel kontrol

CVAT'a yüklemeden önce ön-etiketlerin makul olduğunu gözle doğrulayalım. En kalabalık kareleri
seçiyoruz, çünkü hata (özellikle yanlış pozitif) orada en görünür.

In [ ]:
import matplotlib.pyplot as plt

from tiled_dino import draw_detections

PREVIEW_DIR = os.path.join(OUT_ROOT, "preview")
os.makedirs(PREVIEW_DIR, exist_ok=True)

# Her videodan en kalabalik 2 kare
preview = []
for video in TEST_VIDEOS:
    subset = [r for r in frame_records if r["video"] == video]
    preview.extend(sorted(subset, key=lambda r: -len(r["boxes"]))[:2])

fig, axes = plt.subplots(len(preview), 1, figsize=(15, 8 * len(preview)))
for ax, record in zip(np.atleast_1d(axes), preview):
    frame = cv2.imread(os.path.join(IMAGES_DIR, record["file_name"]))
    annotated = draw_detections(frame, record["boxes"], record["scores"])
    cv2.imwrite(os.path.join(PREVIEW_DIR, record["file_name"]), annotated)
    ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    ax.set_title(f"{record['file_name']}  |  dosemeli {len(record['boxes'])} kutu  "
                 f"(tam kare {record['fullframe_count']})", fontsize=11)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 9. Sıradaki adım: CVAT'ta elle düzeltme

Sağdaki **Output** panelinden şu iki dosyayı indir:

- `test_images.zip` — CVAT'a yüklenecek kareler
- `dataset/test/annotations/test_preannot_cvat.zip` — ön-etiketler

Ayrıca `test_preannot_coco.json` ve `frame_index.json` dosyalarını da indir; değerlendirme
altyapısı bunları kullanacak.

### CVAT akışı (app.cvat.ai, kurulum gerektirmez)

1. **Tasks > Create new task**, ad: `drone_person_test_gt`
2. **Labels > Add label**: `person` (rectangle)
3. **Select files**: `test_images.zip` içindeki görselleri yükle
4. Task oluştuktan sonra **Actions > Upload annotations > CVAT 1.1** ve
   `test_preannot_cvat.zip` dosyasını seç
5. Kareleri sırayla gez:
   - İnsan olmayan kutuları **sil** (kaya, gölge, çanta, ağaç kümesi)
   - Öğretmenin kaçırdığı insanları **ekle**
   - Kutu sınırlarını gerektiği kadar düzelt
6. Bitince **Actions > Export task dataset > COCO 1.0** ile indir

### Ölçmemiz gereken şey: ön-etiketleme yanlılığı

Düzeltme sırasında **eklemek zorunda kaldığın kutu sayısını** not et. Ön-etiket kaynaklı
yanlılığı ölçmek için ayrıca **30 kareyi sıfırdan** (ön-etiketlere bakmadan) etiketle ve
ön-etiketin bu karelerde kaç kutu kaçırdığını raporla. Bu sayı olmadan jüri, GT'nin baseline'a
yanlı olduğu iddiasını haklı olarak öne sürebilir.

### İş yükü beklentisi

~100 kare, çoğunlukla silme/düzeltme işi olduğu için yaklaşık 45-60 dakika.